# 07 — NILMFormer Baseline

**Architecture:** NILMFormer (Petralia et al. KDD 2025)

**Role:** Upper bound reference + Knowledge distillation teacher

**Dataset:** UK-DALE House 1 (85% train / 15% val)

**Author:** Chadha Jeddi — NILM Benchmarking Project

## 1. Setup

In [ ]:
import os, sys, glob

REPO_DIR = '/kaggle/working/nilm-benchmarking'
SAVE_DIR = '/kaggle/working/nilm_results'
CKPT_DIR = f'{SAVE_DIR}/checkpoints'
RES_DIR  = f'{SAVE_DIR}/results'

for d in [SAVE_DIR, CKPT_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
for p in ['src','models','models/baselines','models/proposed']:
    sys.path.insert(0, f'{REPO_DIR}/{p}')

for d in ['data/raw/UKDALE','data/raw/UK-DALE','data/processed',
          'experiments/checkpoints','experiments/results']:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

ukdale_files = glob.glob('/kaggle/input/**/ukdale.h5', recursive=True)
if ukdale_files:
    for dst in ['data/raw/UKDALE/ukdale.h5','data/raw/UK-DALE/ukdale.h5']:
        if not os.path.exists(dst): os.symlink(ukdale_files[0], dst)
    print(f'OK UK-DALE: {ukdale_files[0]}')
else:
    print('ERROR: ukdale.h5 not found')

# Delete old cache to use new Kelly labels
for f in glob.glob(f'{REPO_DIR}/data/processed/*.parquet'):
    os.remove(f); print(f'Deleted: {f}')

os.system('pip install -q PyWavelets pyarrow h5py tqdm einops omegaconf torchinfo seaborn')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Save dir: {SAVE_DIR}')


## 2. Imports & Configuration

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time, json, warnings, os, sys
warnings.filterwarnings('ignore')
os.environ['PYTHONUNBUFFERED'] = '1'

import torch, torch.nn as nn, torch.nn.functional as F

from config import (WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES,
                    APPLIANCE_NAMES, APPLIANCES)
from preprocessing import load_ukdale_house, preprocess_house
from dataset import load_clean_df, save_clean_df, split_train_val, build_dataloaders
from metrics import MetricsTracker, focal_loss
from train import EarlyStopping
from nilmformer_model import NILMFormerModel as ModelClass

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_NAME   = 'nilmformer'
LR           = 1e-4      # NILMFormer standard
WEIGHT_DECAY = 1e-2      # NILMFormer standard
EPOCHS       = 100
PATIENCE     = 25
SCHED_PAT    = 15
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

COLORS = {'kettle':'#D94040','fridge':'#2E9E5A',
          'washing_machine':'#E8922A','dishwasher':'#7B4FBF','microwave':'#CC3399'}
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print(f'Model: {MODEL_NAME} | LR={LR} | WD={WEIGHT_DECAY}')


## 3. Data Loading

In [ ]:
print('Loading UK-DALE House 1...')
cached = load_clean_df('UK-DALE', 1)
if cached is not None:
    clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

print(f'Shape: {clean_df.shape}')
train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

train_loader, val_loader, norm_stats = build_dataloaders(
    train_df, val_df,
    batch_size=256, train_stride=30, val_stride=480,
    num_workers=2, instance_norm=True, seg_size=None,
)

x_s, yp_s, ys_s = next(iter(val_loader))
print(f'Batch: x={tuple(x_s.shape)} y_power={tuple(yp_s.shape)}')

print('\nAppliance duty cycles:')
for a in APPLIANCE_NAMES:
    duty = train_df[f'{a}_state'].mean()*100
    print(f'  {a:<20} duty={duty:.2f}%')


## 4. Model Architecture

In [ ]:
from torchinfo import summary

model = ModelClass(
    in_channels=INPUT_CHANNELS,
    window_size=WINDOW_SIZE,
    n_appliances=N_APPLIANCES,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} | INT8: {n_params/1e6:.3f} MB')
summary(model, input_size=(1, INPUT_CHANNELS, WINDOW_SIZE),
        col_names=['input_size','output_size','num_params'], depth=3)


## 5. Training

In [ ]:
CKPT_PATH = f'{CKPT_DIR}/{MODEL_NAME}_best.pth'
HIST_PATH = f'{RES_DIR}/{MODEL_NAME}_history.json'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=SCHED_PAT)
early_stop = EarlyStopping(patience=PATIENCE)

app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
history = {'epoch':[],'train_loss':[],'val_loss':[],
           'val_mr':[],'val_f1':[],'val_mae':[],'lr':[]}
best_mr, best_state, best_epoch = -float('inf'), None, 0
start_time = time.time()

print(f'Training {MODEL_NAME.upper()} | {n_params:,} params', flush=True)
print(f'LR={LR} | WD={WEIGHT_DECAY}', flush=True)
print('='*80, flush=True)

for epoch in range(1, EPOCHS+1):
    ep_start = time.time()

    # ---- TRAIN ----
    model.train()
    total_train_loss = 0.0; n_batches = 0
    for x, y_power, y_state in train_loader:
        x = x.to(DEVICE)
        y_power = y_power.to(DEVICE)
        y_state = y_state.to(DEVICE)

        pred_p, pred_s, pred_g = model(x)

        loss_power = F.smooth_l1_loss(pred_p, y_power)
        loss_state = focal_loss(pred_s, y_state, alpha=0.75, gamma=2.0)
        loss_gated = F.smooth_l1_loss(pred_g, y_power)
        loss = 1.0*loss_power + 2.0*loss_state + 0.5*loss_gated

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item(); n_batches += 1
    train_loss = total_train_loss / max(n_batches, 1)

    # ---- VALIDATE ----
    model.eval()
    total_val_loss = 0.0; v_batches = 0
    tracker.reset()
    with torch.no_grad():
        for x, yp, ys in val_loader:
            x = x.to(DEVICE); yp = yp.to(DEVICE); ys = ys.to(DEVICE)
            pred_p, pred_s, pred_g = model(x)
            loss = (F.smooth_l1_loss(pred_p, yp)
                    + 2.0*focal_loss(pred_s, ys, alpha=0.75, gamma=2.0)
                    + 0.5*F.smooth_l1_loss(pred_g, yp))
            total_val_loss += loss.item(); v_batches += 1
            tracker.update(pred_p.cpu(), yp.cpu(), pred_s.cpu(), ys.cpu())

    val_loss = total_val_loss / max(v_batches, 1)
    metrics  = tracker.compute()
    val_mr   = metrics['mean']['mr']
    val_f1   = metrics['mean']['f1']
    val_mae  = metrics['mean']['mae_w']
    cur_lr   = optimizer.param_groups[0]['lr']
    ep_time  = time.time() - ep_start

    for k,v in zip(
        ['epoch','train_loss','val_loss','val_mr','val_f1','val_mae','lr'],
        [epoch, train_loss, val_loss, val_mr, val_f1, val_mae, cur_lr]
    ):
        history[k].append(v)

    is_best = val_mr > best_mr
    if is_best:
        best_mr = val_mr; best_epoch = epoch
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        torch.save({
            'model_name': MODEL_NAME,
            'model_state_dict': best_state,
            'best_epoch': best_epoch,
            'best_val_mr': best_mr,
            'n_params': n_params,
            'norm_stats': {
                'agg_mean': norm_stats.agg_mean,
                'agg_std': norm_stats.agg_std,
                'appliance_max': norm_stats.appliance_max,
            },
        }, CKPT_PATH)

    if epoch % 5 == 0 or is_best:
        with open(HIST_PATH, 'w') as f:
            json.dump({**history,
                'best_epoch': best_epoch, 'best_val_mr': best_mr,
                'model': MODEL_NAME,
                'training_time_seconds': time.time()-start_time}, f)

    star = ' *' if is_best else ''
    print(f'Ep {epoch:3d}/{EPOCHS} | L:{train_loss:.4f}/{val_loss:.4f} | '
          f'MR:{val_mr:.3f} F1:{val_f1:.3f} MAE:{val_mae:.1f}W | '
          f'LR:{cur_lr:.1e} | {ep_time:.0f}s{star}', flush=True)

    scheduler.step(val_mr)
    if early_stop.step(val_mr):
        print(f'Early stopping at epoch {epoch}', flush=True); break

total_time = time.time() - start_time
print(f'\nDone in {total_time/60:.1f} min | Best epoch: {best_epoch} | Best MR: {best_mr:.4f}', flush=True)
print(f'Checkpoint: {CKPT_PATH}', flush=True)


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs_list = history['epoch']

axes[0,0].plot(epochs_list, history['train_loss'], color='#3366CC', label='Train')
axes[0,0].plot(epochs_list, history['val_loss'], color='#D94040', label='Val')
axes[0,0].axvline(best_epoch, color='gray', linestyle=':')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')

axes[0,1].plot(epochs_list, history['val_mr'], color='#2E9E5A', linewidth=2)
axes[0,1].axvline(best_epoch, color='gray', linestyle=':')
axes[0,1].axhline(best_mr, color='#2E9E5A', linestyle='--', alpha=0.4,
                  label=f'Best MR={best_mr:.3f}')
axes[0,1].set_title('Matching Ratio'); axes[0,1].legend(); axes[0,1].set_xlabel('Epoch')

axes[1,0].plot(epochs_list, history['val_f1'], color='#E8922A', linewidth=2)
axes[1,0].axvline(best_epoch, color='gray', linestyle=':')
axes[1,0].set_title('F1 Score'); axes[1,0].set_xlabel('Epoch')

axes[1,1].plot(epochs_list, history['val_mae'], color='#7B4FBF', linewidth=2)
axes[1,1].axvline(best_epoch, color='gray', linestyle=':')
axes[1,1].set_title('MAE (Watts)'); axes[1,1].set_xlabel('Epoch')

plt.suptitle(f'NILMFormer — Training Curves (Best: ep{best_epoch} MR={best_mr:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Final Evaluation

In [ ]:
model.load_state_dict(best_state)
model.to(DEVICE); model.eval()

tracker2 = MetricsTracker(APPLIANCE_NAMES, app_max)
tracker2.reset()
all_pp, all_tp, all_ps, all_ts = [], [], [], []

with torch.no_grad():
    for x, yp, ys in val_loader:
        pred_p, pred_s, pred_g = model(x.to(DEVICE))
        tracker2.update(pred_p.cpu(), yp, pred_s.cpu(), ys)
        all_pp.append(pred_p.cpu().numpy())
        all_tp.append(yp.numpy())
        all_ps.append((torch.sigmoid(pred_s.cpu())>=0.5).float().numpy())
        all_ts.append(ys.numpy())

final_metrics = tracker2.compute()
pred_power = np.concatenate(all_pp)
true_power = np.concatenate(all_tp)
pred_state = np.concatenate(all_ps)
true_state = np.concatenate(all_ts)

print(f'\n{"="*80}')
print(f'FINAL RESULTS — NILMFormer (epoch {best_epoch})')
print(f'{"="*80}')
tracker2.print_table(final_metrics)

tracker2.save_json(f'{RES_DIR}/{MODEL_NAME}_metrics.json',
                   final_metrics, model_name=MODEL_NAME)
tracker2.to_dataframe(final_metrics).to_csv(
    f'{RES_DIR}/{MODEL_NAME}_metrics.csv', index=False)


## 8. Disaggregation Visualization

In [ ]:
N_SHOW = 1500
best_start, best_active = 0, 0
for s in range(0, min(len(true_power)-N_SHOW, 50000), N_SHOW):
    active = sum(true_state[s:s+N_SHOW,i].max()>0 for i in range(N_APPLIANCES))
    if active > best_active:
        best_active = active; best_start = s
        if active == N_APPLIANCES: break

val_agg = val_df['aggregate'].values
agg_win = val_agg[best_start:best_start+N_SHOW]
t = range(best_start, best_start+N_SHOW)

fig, axes = plt.subplots(N_APPLIANCES, 1, figsize=(16,3.5*N_APPLIANCES), sharex=True)
for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i]
    max_w = APPLIANCES[a]['max_power']
    tw = true_power[best_start:best_start+N_SHOW,i]*max_w
    pw = pred_power[best_start:best_start+N_SHOW,i]*max_w
    ax.plot(t, agg_win, color='#888', linewidth=0.6, alpha=0.5, label='aggregate')
    ax.plot(t, tw, color='#D94040', linewidth=1.2, label='ground truth')
    ax.plot(t, pw, color='#2196F3', linewidth=1.2, label='predicted')
    ax.set_ylabel(f'{a.replace("_"," ").title()}\n(W)', fontsize=10)
    mr_v=final_metrics[a]['mr']; f1_v=final_metrics[a]['f1']; mae_v=final_metrics[a]['mae_w']
    ax.text(0.01,0.92,f'MR={mr_v:.3f}  F1={f1_v:.3f}  MAE={mae_v:.1f}W',
            transform=ax.transAxes,fontsize=9,
            bbox=dict(boxstyle='round',facecolor='white',alpha=0.85))
    if i==0: ax.legend(loc='upper right',fontsize=8,ncol=3)
    if i<N_APPLIANCES-1: ax.set_xticklabels([])
axes[-1].set_xlabel('Timestep (x6 seconds)')
plt.suptitle('NILMFormer — Disaggregation Results', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_disaggregation.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(4*N_APPLIANCES, 4))
for i, a in enumerate(APPLIANCE_NAMES):
    cm = confusion_matrix(true_state[:,i], pred_state[:,i], labels=[0,1])
    cm_norm = cm.astype('float')/(cm.sum(axis=1,keepdims=True)+1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=['OFF','ON'], yticklabels=['OFF','ON'],
                ax=axes[i], cbar=i==N_APPLIANCES-1, vmin=0, vmax=1)
    f1_v=final_metrics[a]['f1']; prec=final_metrics[a]['precision']; rec=final_metrics[a]['recall']
    axes[i].set_title(f'{a.replace("_"," ").title()}\nF1={f1_v:.3f} P={prec:.3f} R={rec:.3f}', fontsize=9)
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Actual' if i==0 else '')
plt.suptitle('NILMFormer — Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = [COLORS[a] for a in APPLIANCE_NAMES]
x_pos = np.arange(N_APPLIANCES)
for ax, metric, label, fmt in zip(axes,
    ['f1','mae_w','mr'],['F1 Score','MAE (W)','Matching Ratio'],
    ['{:.3f}','{:.1f}','{:.3f}']):
    vals = [final_metrics[a][metric] for a in APPLIANCE_NAMES]
    bars = ax.bar(x_pos, vals, color=colors, alpha=0.85)
    ax.set_xticks(x_pos); ax.set_xticklabels(APPLIANCE_NAMES, rotation=25, ha='right')
    ax.set_title(label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.02, fmt.format(val), ha='center', fontsize=9)
plt.suptitle('NILMFormer — Per-Appliance Results', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. House 2 Evaluation (thesis number)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dwt import dwt_transform

class House2Dataset(Dataset):
    def __init__(self, df, stride=480):
        self.df = df
        self.starts = list(range(0, len(df)-WINDOW_SIZE+1, stride))
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]; c = s + WINDOW_SIZE//2
        window = self.df['aggregate'].values[s:s+WINDOW_SIZE].astype(np.float32)
        w_mean = window.mean(); w_std = window.std() + 1e-8
        x = dwt_transform((window-w_mean)/w_std)
        hour = self.df.index[c].hour
        sin_h = np.full(WINDOW_SIZE, np.sin(2*np.pi*hour/24), dtype=np.float32)
        cos_h = np.full(WINDOW_SIZE, np.cos(2*np.pi*hour/24), dtype=np.float32)
        x = np.concatenate([x, sin_h[None], cos_h[None]], axis=0)
        y_power = np.array([np.nan_to_num(self.df[a].values[c]/APPLIANCES[a]['max_power'],nan=0.0)
            for a in APPLIANCE_NAMES], dtype=np.float32).clip(0,1)
        y_state = np.array([np.nan_to_num(self.df[f'{a}_state'].values[c],nan=0.0)
            for a in APPLIANCE_NAMES], dtype=np.float32)
        return torch.tensor(x), torch.tensor(y_power), torch.tensor(y_state)

print('Loading House 2...')
cached2 = load_clean_df('UK-DALE', 2)
if cached2 is not None:
    h2_df = cached2
else:
    raw2 = load_ukdale_house(house=2)
    h2_df = preprocess_house(raw2)
    save_clean_df(h2_df, 'UK-DALE', 2)
print(f'House 2: {len(h2_df):,} rows')

h2_loader = DataLoader(House2Dataset(h2_df, stride=480),
                       batch_size=256, shuffle=False, num_workers=2)

model.load_state_dict(best_state); model.to(DEVICE); model.eval()
h2_tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
h2_tracker.reset()
with torch.no_grad():
    for x, yp, ys in h2_loader:
        pred_p, pred_s, pred_g = model(x.to(DEVICE))
        h2_tracker.update(pred_p.cpu(), yp, pred_s.cpu(), ys)

h2_metrics = h2_tracker.compute()
print(f'\n{"="*70}')
print(f'HOUSE 2 RESULTS (unseen) — NILMFormer')
print(f'{"="*70}')
h2_tracker.print_table(h2_metrics)

with open(f'{RES_DIR}/{MODEL_NAME}_h2_metrics.json','w') as f:
    json.dump(h2_metrics, f, indent=2)
print('Saved: h2_metrics.json')


## 12. Save All

In [ ]:
norm_stats.save(f'{RES_DIR}/{MODEL_NAME}_norm_stats.json')
with open(HIST_PATH, 'w') as f:
    json.dump({**history,
        'best_epoch': best_epoch, 'best_val_mr': best_mr,
        'model': MODEL_NAME,
        'training_time_seconds': total_time}, f, indent=2)

print(f'All results in: {RES_DIR}')
for f in sorted(os.listdir(RES_DIR)):
    if MODEL_NAME in f:
        size = os.path.getsize(f'{RES_DIR}/{f}')/1e6
        print(f'  {f}: {size:.1f} MB')
print(f'\nNILMFormer | Best MR: {best_mr:.4f} | Epoch: {best_epoch}')
print(f'Training time: {total_time/60:.1f} min')
